In [ ]:
"""
═══════════════════════════════════════════════════════════════════
SLU INTERNATIONAL STUDENT APPLICATIONS - WEEK 1
Notebook 2: Data Quality Assessment
═══════════════════════════════════════════════════════════════════

Purpose: Identify and document data quality issues including duplicates,
         missing values, relationship integrity, and anomalies

Author: [Your Name]
Date: February 2026
Week: 1 - Data Foundation & Exploration

Focus Areas:
- Duplicate detection (especially Connect data)
- Cross-file relationship integrity
- Business logic validation
- Anomaly detection
- Email validation

═══════════════════════════════════════════════════════════════════
"""

# ═══════════════════════════════════════════════════════════════════
# SECTION 1: ENVIRONMENT SETUP
# ═══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Environment setup complete")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ═══════════════════════════════════════════════════════════════════
# SECTION 2: LOAD DATA
# ═══════════════════════════════════════════════════════════════════

print("LOADING DATASETS")
print("═" * 70)

# ───────────────────────────────────────────────────────────────────
# TODO: Upload the same 4 CSV files from Notebook 1
# ───────────────────────────────────────────────────────────────────

from google.colab import files

print("Please upload your 4 CSV files:")
uploaded = files.upload()

# Auto-detect filenames
def find_file(keyword, uploaded_files):
    for filename in uploaded_files:
        if keyword.lower() in filename.lower():
            return filename
    return None

uploaded_files = list(uploaded.keys())
APPLICANT_FILE = find_file('Applicant', uploaded_files)
SEVIS_FILE = find_file('SEVIS', uploaded_files)
CONNECT_FILE = find_file('Connect', uploaded_files)
OUTREACH_FILE = find_file('Outreach', uploaded_files)

print(f"\n✓ Detected files:")
print(f"  Applicant → {APPLICANT_FILE}")
print(f"  SEVIS     → {SEVIS_FILE}")
print(f"  Connect   → {CONNECT_FILE}")
print(f"  Outreach  → {OUTREACH_FILE}\n")

# Load datasets
applicant_df = pd.read_csv(APPLICANT_FILE, low_memory=False)
sevis_df = pd.read_csv(SEVIS_FILE, low_memory=False)
connect_df = pd.read_csv(CONNECT_FILE, low_memory=False)
outreach_df = pd.read_csv(OUTREACH_FILE, low_memory=False)

print(f"✓ Loaded Applicant: {applicant_df.shape[0]:,} rows × {applicant_df.shape[1]} columns")
print(f"✓ Loaded SEVIS:     {sevis_df.shape[0]:,} rows × {sevis_df.shape[1]} columns")
print(f"✓ Loaded Connect:   {connect_df.shape[0]:,} rows × {connect_df.shape[1]} columns")
print(f"✓ Loaded Outreach:  {outreach_df.shape[0]:,} rows × {outreach_df.shape[1]} columns")
print()

✓ Environment setup complete
✓ Timestamp: 2026-02-08 20:22:29

LOADING DATASETS
══════════════════════════════════════════════════════════════════════
Please upload your 4 CSV files:


Saving Mein Copy of Connect - Sheet1.csv to Mein Copy of Connect - Sheet1.csv
Saving Mein Copy of Outreach_Data - Sheet1.csv to Mein Copy of Outreach_Data - Sheet1.csv
Saving Mein Copy of SEVIS - Sheet1.csv to Mein Copy of SEVIS - Sheet1.csv
Saving Mien Copy of Applicant_Data - Sheet1.csv to Mien Copy of Applicant_Data - Sheet1.csv

✓ Detected files:
  Applicant → Mien Copy of Applicant_Data - Sheet1.csv
  SEVIS     → Mein Copy of SEVIS - Sheet1.csv
  Connect   → Mein Copy of Connect - Sheet1.csv
  Outreach  → Mein Copy of Outreach_Data - Sheet1.csv

✓ Loaded Applicant: 6,894 rows × 30 columns
✓ Loaded SEVIS:     3,852 rows × 123 columns
✓ Loaded Connect:   34,341 rows × 8 columns
✓ Loaded Outreach:  1,565 rows × 18 columns



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 3: DUPLICATE DETECTION
# ═══════════════════════════════════════════════════════════════════

print("DUPLICATE RECORD ANALYSIS")
print("═" * 70)

# ───────────────────────────────────────────────────────────────────
# TODO: Check for duplicate IDs in each dataset
# Focus on Reference_ID and SEVIS_ID columns
# ───────────────────────────────────────────────────────────────────

def analyze_duplicates(df, id_column, dataset_name):
    """
    Analyze duplicate IDs in a dataset

    Parameters:
    -----------
    df : pd.DataFrame
        Dataset to analyze
    id_column : str
        Name of ID column to check
    dataset_name : str
        Name of dataset for display

    Returns:
    --------
    dict with duplicate statistics
    """
    if id_column not in df.columns:
        return None

    total_records = len(df)
    non_null_ids = df[id_column].count()
    unique_ids = df[id_column].nunique()
    duplicate_count = non_null_ids - unique_ids
    duplicate_pct = (duplicate_count / non_null_ids * 100) if non_null_ids > 0 else 0
    null_count = df[id_column].isna().sum()

    return {
        'Dataset': dataset_name,
        'ID_Column': id_column,
        'Total_Records': total_records,
        'Non_Null_IDs': non_null_ids,
        'Unique_IDs': unique_ids,
        'Duplicate_IDs': duplicate_count,
        'Duplicate_%': round(duplicate_pct, 2),
        'Null_IDs': null_count,
        'Null_%': round(null_count / total_records * 100, 2)
    }

# Analyze duplicates in each dataset
duplicate_results = []

# Applicant - Reference_ID
result = analyze_duplicates(applicant_df, 'Reference_ID', 'Applicant')
if result:
    duplicate_results.append(result)

# Applicant - SEVIS_ID
result = analyze_duplicates(applicant_df, 'SEVIS_ID', 'Applicant')
if result:
    duplicate_results.append(result)

# SEVIS - SEVIS_ID
result = analyze_duplicates(sevis_df, 'SEVIS_ID', 'SEVIS')
if result:
    duplicate_results.append(result)

# Connect - Reference_ID
result = analyze_duplicates(connect_df, 'Reference_ID', 'Connect')
if result:
    duplicate_results.append(result)

# Outreach - SEVIS_ID
result = analyze_duplicates(outreach_df, 'SEVIS_ID', 'Outreach')
if result:
    duplicate_results.append(result)

# Create results DataFrame
duplicate_df = pd.DataFrame(duplicate_results)
print(duplicate_df.to_string(index=False))

# ───────────────────────────────────────────────────────────────────
# Highlight critical findings
# ───────────────────────────────────────────────────────────────────

print("\n" + "═" * 70)
print("🔍 KEY FINDINGS:")
print("═" * 70)

# Find the worst offender
max_dup = duplicate_df.loc[duplicate_df['Duplicate_%'].idxmax()]
print(f"\n⚠ CRITICAL: {max_dup['Dataset']} - {max_dup['ID_Column']}")
print(f"  {max_dup['Duplicate_IDs']:,} duplicate IDs ({max_dup['Duplicate_%']:.2f}%)")
print(f"  {max_dup['Total_Records']:,} total records but only {max_dup['Unique_IDs']:,} unique IDs")
print()
print("  💡 HYPOTHESIS: Connect data is an EVENT LOG, not a status table.")
print("     Each row may represent an interaction/update timestamp.")
print("     Students appear multiple times as their status changes.")

# Check for any perfect duplicates (all columns identical)
print("\n\n" + "─" * 70)
print("CHECKING FOR PERFECT DUPLICATES (entire row identical)")
print("─" * 70)

for name, df in [('Applicant', applicant_df), ('SEVIS', sevis_df),
                 ('Connect', connect_df), ('Outreach', outreach_df)]:
    perfect_dupes = df.duplicated().sum()
    if perfect_dupes > 0:
        print(f"⚠ {name}: {perfect_dupes:,} perfect duplicate rows found")
    else:
        print(f"✓ {name}: No perfect duplicates")

DUPLICATE RECORD ANALYSIS
══════════════════════════════════════════════════════════════════════
  Dataset    ID_Column  Total_Records  Non_Null_IDs  Unique_IDs  Duplicate_IDs  Duplicate_%  Null_IDs  Null_%
Applicant Reference_ID           6894          6894        6893              1         0.01         0     0.0
Applicant     SEVIS_ID           6894          2606        2605              1         0.04      4288    62.2
    SEVIS     SEVIS_ID           3852          3852        3852              0         0.00         0     0.0
  Connect Reference_ID          34341         34341        6875          27466        79.98         0     0.0
 Outreach     SEVIS_ID           1565          1565        1561              4         0.26         0     0.0

══════════════════════════════════════════════════════════════════════
🔍 KEY FINDINGS:
══════════════════════════════════════════════════════════════════════

⚠ CRITICAL: Connect - Reference_ID
  27,466 duplicate IDs (79.98%)
  34,341 total r

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 4: DUPLICATE ID DEEP DIVE - CONNECT DATA
# ═══════════════════════════════════════════════════════════════════

print("\n\nDEEP DIVE: CONNECT DATA DUPLICATION PATTERN")
print("═" * 70)

# ───────────────────────────────────────────────────────────────────
# TODO: Analyze how students appear multiple times in Connect
# Show example of a student with many records
# ───────────────────────────────────────────────────────────────────

# Count how many times each Reference_ID appears
ref_id_counts = connect_df['Reference_ID'].value_counts()

print(f"Distribution of student appearances in Connect:")
print(f"  Students appearing once:      {(ref_id_counts == 1).sum():,}")
print(f"  Students appearing 2-5 times: {((ref_id_counts >= 2) & (ref_id_counts <= 5)).sum():,}")
print(f"  Students appearing 6-10 times: {((ref_id_counts >= 6) & (ref_id_counts <= 10)).sum():,}")
print(f"  Students appearing >10 times:  {(ref_id_counts > 10).sum():,}")
print(f"  Maximum appearances:          {ref_id_counts.max()}")
print(f"  Average appearances:          {ref_id_counts.mean():.1f}")

# Show example student with multiple records
print("\n" + "─" * 70)
print("EXAMPLE: Student with Multiple Records")
print("─" * 70)

# Find a student with many records
frequent_student = ref_id_counts.index[0]  # Student with most records
student_records = connect_df[connect_df['Reference_ID'] == frequent_student].copy()

print(f"\nStudent Reference_ID: {frequent_student}")
print(f"Number of records: {len(student_records)}")
print(f"\nAll records for this student:")

# Show relevant columns
display_cols = ['Reference_ID', 'Deposit_Status', 'I_20_Status',
                'Recieved_At', 'Created_At']
# Only show columns that exist
display_cols = [col for col in display_cols if col in student_records.columns]

print(student_records[display_cols].to_string(index=False))

print("\n💡 OBSERVATION:")
print("   Multiple records with same or changing status over time.")
print("   This confirms Connect is tracking EVENTS/INTERACTIONS, not final status.")



DEEP DIVE: CONNECT DATA DUPLICATION PATTERN
══════════════════════════════════════════════════════════════════════
Distribution of student appearances in Connect:
  Students appearing once:      11
  Students appearing 2-5 times: 6,855
  Students appearing 6-10 times: 9
  Students appearing >10 times:  0
  Maximum appearances:          7
  Average appearances:          5.0

──────────────────────────────────────────────────────────────────────
EXAMPLE: Student with Multiple Records
──────────────────────────────────────────────────────────────────────

Student Reference_ID: 1319991
Number of records: 7

All records for this student:
 Reference_ID Deposit_Status I_20_Status     Recieved_At   Created_At
      1319991            Yes         Yes 4/14/2025 19:55 1.740000e+12
      1319991            Yes         Yes 4/16/2025 16:06 1.740000e+12
      1319991            Yes         Yes 5/27/2025 11:58 1.750000e+12
      1319991            Yes         Yes 5/27/2025 12:00 1.750000e+12
      1

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 5: CROSS-FILE RELATIONSHIP INTEGRITY
# ═══════════════════════════════════════════════════════════════════

print("\n\nCROSS-FILE RELATIONSHIP INTEGRITY CHECKS")
print("═" * 70)

# ───────────────────────────────────────────────────────────────────
# TODO: Check if IDs in one dataset exist in related datasets
# This validates data consistency across files
# ───────────────────────────────────────────────────────────────────

integrity_results = []

# ───────────────────────────────────────────────────────────────────
# CHECK 1: Applicant SEVIS_IDs should exist in SEVIS data
# ───────────────────────────────────────────────────────────────────

applicant_sevis_ids = set(applicant_df['SEVIS_ID'].dropna())
sevis_ids = set(sevis_df['SEVIS_ID'].dropna())

missing_in_sevis = applicant_sevis_ids - sevis_ids
match_count = len(applicant_sevis_ids) - len(missing_in_sevis)
match_rate = (match_count / len(applicant_sevis_ids) * 100) if len(applicant_sevis_ids) > 0 else 0

integrity_results.append({
    'Check': 'Applicant SEVIS_IDs → SEVIS Data',
    'Source_Count': len(applicant_sevis_ids),
    'Target_Count': len(sevis_ids),
    'Matched': match_count,
    'Missing': len(missing_in_sevis),
    'Match_Rate_%': round(match_rate, 2)
})

print(f"\n1. Applicant SEVIS_IDs in SEVIS Data:")
print(f"   Applicant SEVIS_IDs:  {len(applicant_sevis_ids):,}")
print(f"   SEVIS records:        {len(sevis_ids):,}")
print(f"   Matched:              {match_count:,} ({match_rate:.2f}%)")
print(f"   Missing in SEVIS:     {len(missing_in_sevis):,}")

if match_rate < 50:
    print(f"   ⚠ WARNING: Low match rate! Possible data sync issue.")

# ───────────────────────────────────────────────────────────────────
# CHECK 2: Connect Reference_IDs should exist in Applicant data
# ───────────────────────────────────────────────────────────────────

connect_ref_ids = set(connect_df['Reference_ID'].dropna())
applicant_ref_ids = set(applicant_df['Reference_ID'].dropna())

missing_in_applicant = connect_ref_ids - applicant_ref_ids
match_count = len(connect_ref_ids) - len(missing_in_applicant)
match_rate = (match_count / len(connect_ref_ids) * 100) if len(connect_ref_ids) > 0 else 0

integrity_results.append({
    'Check': 'Connect Reference_IDs → Applicant Data',
    'Source_Count': len(connect_ref_ids),
    'Target_Count': len(applicant_ref_ids),
    'Matched': match_count,
    'Missing': len(missing_in_applicant),
    'Match_Rate_%': round(match_rate, 2)
})

print(f"\n2. Connect Reference_IDs in Applicant Data:")
print(f"   Connect Reference_IDs: {len(connect_ref_ids):,}")
print(f"   Applicant records:     {len(applicant_ref_ids):,}")
print(f"   Matched:               {match_count:,} ({match_rate:.2f}%)")
print(f"   Missing in Applicant:  {len(missing_in_applicant):,}")

if match_rate == 100:
    print(f"   ✓ PERFECT: All Connect students found in Applicant!")

# ───────────────────────────────────────────────────────────────────
# CHECK 3: Outreach SEVIS_IDs should exist in SEVIS or Applicant
# ───────────────────────────────────────────────────────────────────

outreach_sevis_ids = set(outreach_df['SEVIS_ID'].dropna())
all_sevis_ids = sevis_ids.union(applicant_sevis_ids)

missing_in_both = outreach_sevis_ids - all_sevis_ids
match_count = len(outreach_sevis_ids) - len(missing_in_both)
match_rate = (match_count / len(outreach_sevis_ids) * 100) if len(outreach_sevis_ids) > 0 else 0

integrity_results.append({
    'Check': 'Outreach SEVIS_IDs → SEVIS/Applicant Data',
    'Source_Count': len(outreach_sevis_ids),
    'Target_Count': len(all_sevis_ids),
    'Matched': match_count,
    'Missing': len(missing_in_both),
    'Match_Rate_%': round(match_rate, 2)
})

print(f"\n3. Outreach SEVIS_IDs in SEVIS/Applicant Data:")
print(f"   Outreach SEVIS_IDs:    {len(outreach_sevis_ids):,}")
print(f"   Combined SEVIS IDs:    {len(all_sevis_ids):,}")
print(f"   Matched:               {match_count:,} ({match_rate:.2f}%)")
print(f"   Missing in both:       {len(missing_in_both):,}")

# Create summary table
print("\n" + "═" * 70)
print("RELATIONSHIP INTEGRITY SUMMARY")
print("═" * 70)
integrity_df = pd.DataFrame(integrity_results)
print(integrity_df.to_string(index=False))



CROSS-FILE RELATIONSHIP INTEGRITY CHECKS
══════════════════════════════════════════════════════════════════════

1. Applicant SEVIS_IDs in SEVIS Data:
   Applicant SEVIS_IDs:  2,605
   SEVIS records:        3,852
   Matched:              615 (23.61%)
   Missing in SEVIS:     1,990
   ⚠ WARNING: Low match rate! Possible data sync issue.

2. Connect Reference_IDs in Applicant Data:
   Connect Reference_IDs: 6,875
   Applicant records:     6,893
   Matched:               6,875 (100.00%)
   Missing in Applicant:  0
   ✓ PERFECT: All Connect students found in Applicant!

3. Outreach SEVIS_IDs in SEVIS/Applicant Data:
   Outreach SEVIS_IDs:    1,561
   Combined SEVIS IDs:    5,842
   Matched:               1,560 (99.94%)
   Missing in both:       1

══════════════════════════════════════════════════════════════════════
RELATIONSHIP INTEGRITY SUMMARY
══════════════════════════════════════════════════════════════════════
                                    Check  Source_Count  Target_Count  

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INVESTIGATION: Who Are the Missing SEVIS_IDs?
# ═══════════════════════════════════════════════════════════════════

print("\n\n" + "─" * 70)
print("INVESTIGATING THE 1,990 MISSING SEVIS_IDs")
print("─" * 70)

# Find applicants with SEVIS_IDs that don't exist in SEVIS data
missing_sevis_mask = (
    applicant_df['SEVIS_ID'].notna() &  # Has SEVIS_ID
    ~applicant_df['SEVIS_ID'].isin(sevis_ids)  # But not in SEVIS dataset
)

missing_sevis_students = applicant_df[missing_sevis_mask].copy()

print(f"\nTotal missing: {len(missing_sevis_students):,}")
print(f"\nAnalyzing their characteristics...")

# Check their intake terms
print("\n1. INTAKE DISTRIBUTION (Missing students):")
print("─" * 50)
intake_dist = missing_sevis_students['Intake'].value_counts().head(10)
for intake, count in intake_dist.items():
    pct = count / len(missing_sevis_students) * 100
    print(f"  {str(intake):20s}: {count:4d} ({pct:5.2f}%)")

# Check if they have deposit/I-20 status in Connect
print("\n2. CHECKING CONNECT DATA FOR MISSING STUDENTS:")
print("─" * 50)

# Join with Connect to see their status
missing_ref_ids = missing_sevis_students['Reference_ID'].unique()
connect_for_missing = connect_df[connect_df['Reference_ID'].isin(missing_ref_ids)]

if len(connect_for_missing) > 0:
    # Deduplicate to get latest status
    connect_latest = connect_for_missing.sort_values('Created_At').groupby('Reference_ID').tail(1)

    print(f"  Found in Connect: {len(connect_latest):,} students")

    if 'Deposit_Status' in connect_latest.columns:
        deposit_dist = connect_latest['Deposit_Status'].value_counts()
        print(f"\n  Deposit Status:")
        for status, count in deposit_dist.items():
            pct = count / len(connect_latest) * 100
            print(f"    {status}: {count:4d} ({pct:5.2f}%)")

    if 'I_20_Status' in connect_latest.columns:
        i20_dist = connect_latest['I_20_Status'].value_counts()
        print(f"\n  I-20 Status:")
        for status, count in i20_dist.items():
            pct = count / len(connect_latest) * 100
            print(f"    {status}: {count:4d} ({pct:5.2f}%)")
else:
    print("  Not found in Connect data")

# Check application source
print("\n3. APPLICATION SOURCE (Missing students):")
print("─" * 50)
if 'Application_Source' in missing_sevis_students.columns:
    source_dist = missing_sevis_students['Application_Source'].value_counts().head(5)
    for source, count in source_dist.items():
        pct = count / missing_sevis_students['Application_Source'].count() * 100
        print(f"  {source:30s}: {count:4d} ({pct:5.2f}%)")

print("\n" + "─" * 70)
print("💡 HYPOTHESIS:")
print("─" * 70)
print("The 1,990 missing SEVIS_IDs are likely students who:")
print("  1. Were assigned SEVIS_IDs during admission (provisional)")
print("  2. But never completed enrollment (no deposit/I-20)")
print("  3. Or are from older cohorts not included in SEVIS export")
print("  4. Or SEVIS export only includes 'INITIAL' status students")





──────────────────────────────────────────────────────────────────────
INVESTIGATING THE 1,990 MISSING SEVIS_IDs
──────────────────────────────────────────────────────────────────────

Total missing: 1,991

Analyzing their characteristics...

1. INTAKE DISTRIBUTION (Missing students):
──────────────────────────────────────────────────
  Fall 2025           : 1235 (62.03%)
  Summer 2025         :  397 (19.94%)
  Spring 2025         :  189 ( 9.49%)
  Fall 2 2025         :   51 ( 2.56%)
  Summer 2 2025       :   42 ( 2.11%)
  Fall 2024           :   33 ( 1.66%)
  Spring 2026         :   23 ( 1.16%)
  Summer 2024         :    9 ( 0.45%)
  Spring 2 2025       :    6 ( 0.30%)
  Fall 2 2024         :    3 ( 0.15%)

2. CHECKING CONNECT DATA FOR MISSING STUDENTS:
──────────────────────────────────────────────────
  Found in Connect: 1,991 students

  Deposit Status:
    No: 1138 (57.16%)
    Yes:  853 (42.84%)

  I-20 Status:
    Yes: 1913 (96.08%)
    No:   78 ( 3.92%)

3. APPLICATION SOURCE

The analysis of the 1,990 missing SEVIS_IDs from the Applicant data that are not found in the SEVIS data has completed.

Here are the key findings:

Intake Distribution: A significant majority

*  Intake Distribution: A significant majority of the missing students are from the Fall 2025 (62.03%), Summer 2025 (19.94%), and Spring 2025 (9.49%) intakes, indicating they are recent applicants.

*  Connect Data Status: For these missing students, the Connect data shows:

1.  Deposit Status: 57.16% have 'No' deposit status, while 42.84% have 'Yes'. This suggests a portion of these students did pay a deposit, but their SEVIS ID is still missing in the SEVIS dataset.
2.  I-20 Status: 96.08% have an 'Yes' I-20 status, and only 3.92% have 'No'. This is a critical finding, as it implies that many students who seemingly have an I-20 processed in the Connect system are not reflected in the SEVIS data.



* Application Source: The primary application
sources are Slate (73.88%) and INTO (23.36%).

Based on this, the hypothesis is:
The 1,990 missing SEVIS_IDs are likely students who:

* Were assigned SEVIS_IDs during admission (provisional).
But never completed enrollment (no deposit/I-20).

*  Or are from older cohorts not included in the SEVIS export.
*  Or the SEVIS export only includes 'INITIAL' status students.


Expected Root Cause:

It is likely a combination of factors, primarily a data scope mismatch where the Applicant data includes all historical admissions while SEVIS data only includes currently active I-20s. There's an expected gap from students who don't enroll, and potentially a timing mismatch between data exports. However, the high percentage of 'Yes' I-20 statuses in the Connect data for these missing students is concerning and warrants further investigation.

Recommendation:

It is recommended to verify with stakeholders if the SEVIS export is filtered to active students only, what the expected deposit-to-I-20 conversion rate is, and if the 1,990 missing students are indeed expected non-enrollments or if there's a data synchronization issue, especially given the I-20 status in Connect.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 6: BUSINESS LOGIC VALIDATION
# ═══════════════════════════════════════════════════════════════════

print("\n\nBUSINESS LOGIC VALIDATION")
print("═" * 70)
print("Checking for violations of expected business rules...")

# ───────────────────────────────────────────────────────────────────
# TODO: Validate business logic rules
# Example: Students with deposit should have I-20 issued
# ───────────────────────────────────────────────────────────────────

business_violations = []

# ───────────────────────────────────────────────────────────────────
# RULE 1: Students who paid deposit should have I-20 issued
# ───────────────────────────────────────────────────────────────────

print("\n1. RULE: Deposit Paid → I-20 Should Be Issued")
print("─" * 70)

if 'Deposit_Status' in connect_df.columns and 'I_20_Status' in connect_df.columns:
    # Deduplicate to get latest status per student
    connect_latest = connect_df.sort_values('Created_At').groupby('Reference_ID').tail(1)

    # Find students with deposit but no I-20
    deposit_yes = connect_latest[connect_latest['Deposit_Status'] == 'Yes']
    deposit_yes_no_i20 = deposit_yes[deposit_yes['I_20_Status'] != 'Yes']

    total_deposits = len(deposit_yes)
    violation_count = len(deposit_yes_no_i20)
    violation_pct = (violation_count / total_deposits * 100) if total_deposits > 0 else 0

    print(f"  Total students with deposit: {total_deposits:,}")
    print(f"  Without I-20 issued:         {violation_count:,} ({violation_pct:.2f}%)")

    if violation_pct > 10:
        print(f"  ⚠ HIGH: {violation_pct:.1f}% of deposits haven't resulted in I-20")
        print(f"     Possible processing bottleneck or timing issue")
    elif violation_pct > 0:
        print(f"  ⚠ MODERATE: Some deposits pending I-20 issuance")
    else:
        print(f"  ✓ GOOD: All deposits have I-20s issued")

    business_violations.append({
        'Rule': 'Deposit → I-20 Issued',
        'Total_Records': total_deposits,
        'Violations': violation_count,
        'Violation_%': round(violation_pct, 2),
        'Severity': 'HIGH' if violation_pct > 10 else 'MODERATE' if violation_pct > 0 else 'NONE'
    })

# ───────────────────────────────────────────────────────────────────
# RULE 2: I-20 issued students should have SEVIS record
# ───────────────────────────────────────────────────────────────────

print("\n2. RULE: I-20 Issued → Should Have SEVIS Record")
print("─" * 70)

if 'I_20_Status' in connect_df.columns:
    connect_latest = connect_df.sort_values('Created_At').groupby('Reference_ID').tail(1)

    # Get students with I-20 issued
    i20_issued = connect_latest[connect_latest['I_20_Status'] == 'Yes']

    # Join with applicant to get SEVIS_IDs
    i20_with_applicant = i20_issued.merge(
        applicant_df[['Reference_ID', 'SEVIS_ID']],
        on='Reference_ID',
        how='left'
    )

    # Check how many have SEVIS_ID
    total_i20 = len(i20_issued)
    with_sevis_id = i20_with_applicant['SEVIS_ID'].notna().sum()
    without_sevis_id = total_i20 - with_sevis_id

    print(f"  Total students with I-20:    {total_i20:,}")
    print(f"  Have SEVIS_ID in Applicant:  {with_sevis_id:,}")
    print(f"  Missing SEVIS_ID:            {without_sevis_id:,}")

    if without_sevis_id > 0:
        print(f"  ⚠ WARNING: {without_sevis_id} I-20s issued but no SEVIS_ID recorded")

# ───────────────────────────────────────────────────────────────────
# RULE 3: Visa approved students should have SEVIS record
# ───────────────────────────────────────────────────────────────────

print("\n3. RULE: Visa Approved → Must Have SEVIS_ID")
print("─" * 70)

if 'Student_Status' in outreach_df.columns and 'SEVIS_ID' in outreach_df.columns:
    visa_approved = outreach_df[
        outreach_df['Student_Status'].str.contains('Visa approved', na=False, case=False)
    ]

    total_visa_approved = len(visa_approved)
    with_sevis = visa_approved['SEVIS_ID'].notna().sum()
    without_sevis = total_visa_approved - with_sevis

    print(f"  Total visa approved:         {total_visa_approved:,}")
    print(f"  Have SEVIS_ID:               {with_sevis:,}")
    print(f"  Missing SEVIS_ID:            {without_sevis:,}")

    if without_sevis == 0:
        print(f"  ✓ PERFECT: All visa-approved students have SEVIS_IDs")
    else:
        print(f"  ⚠ CRITICAL: {without_sevis} visa approvals without SEVIS_ID")

    business_violations.append({
        'Rule': 'Visa Approved → SEVIS_ID',
        'Total_Records': total_visa_approved,
        'Violations': without_sevis,
        'Violation_%': round(without_sevis / total_visa_approved * 100, 2) if total_visa_approved > 0 else 0,
        'Severity': 'CRITICAL' if without_sevis > 0 else 'NONE'
    })

# ───────────────────────────────────────────────────────────────────
# Summary of violations
# ───────────────────────────────────────────────────────────────────

if business_violations:
    print("\n" + "═" * 70)
    print("BUSINESS LOGIC VIOLATIONS SUMMARY")
    print("═" * 70)
    violations_df = pd.DataFrame(business_violations)
    print(violations_df.to_string(index=False))



BUSINESS LOGIC VALIDATION
══════════════════════════════════════════════════════════════════════
Checking for violations of expected business rules...

1. RULE: Deposit Paid → I-20 Should Be Issued
──────────────────────────────────────────────────────────────────────
  Total students with deposit: 1,512
  Without I-20 issued:         262 (17.33%)
  ⚠ HIGH: 17.3% of deposits haven't resulted in I-20
     Possible processing bottleneck or timing issue

2. RULE: I-20 Issued → Should Have SEVIS Record
──────────────────────────────────────────────────────────────────────
  Total students with I-20:    2,389
  Have SEVIS_ID in Applicant:  2,387
  Missing SEVIS_ID:            2
  ⚠ WARNING: 2 I-20s issued but no SEVIS_ID recorded

3. RULE: Visa Approved → Must Have SEVIS_ID
──────────────────────────────────────────────────────────────────────
  Total visa approved:         382
  Have SEVIS_ID:               382
  Missing SEVIS_ID:            0
  ✓ PERFECT: All visa-approved students have

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 7: EMAIL VALIDATION
# ═══════════════════════════════════════════════════════════════════

print("\n\nEMAIL ADDRESS VALIDATION")
print("═" * 70)

# ───────────────────────────────────────────────────────────────────
# TODO: Check email format and consistency
# Verify @ symbol, valid domains, SLU email patterns
# ───────────────────────────────────────────────────────────────────

def validate_emails(df, email_col, dataset_name):
    """
    Validate email addresses in a column

    Parameters:
    -----------
    df : pd.DataFrame
        Dataset containing email column
    email_col : str
        Name of email column
    dataset_name : str
        Name of dataset for display

    Returns:
    --------
    dict with validation results
    """
    if email_col not in df.columns:
        return None

    emails = df[email_col].dropna()
    total = len(emails)

    if total == 0:
        return None

    # Check for @ symbol
    has_at = emails.str.contains('@', na=False).sum()

    # Check for common domains
    has_valid_domain = emails.str.contains(
        r'\.(com|edu|org|net|gov)',
        na=False,
        case=False
    ).sum()

    # Check for SLU email
    has_slu = emails.str.contains('slu.edu', na=False, case=False).sum()

    # Potential issues
    issues = total - has_at

    return {
        'Dataset': dataset_name,
        'Email_Column': email_col,
        'Total_Non_Null': total,
        'Has_@_Symbol': has_at,
        'Has_Valid_Domain': has_valid_domain,
        'Is_SLU_Email': has_slu,
        'Potential_Issues': issues,
        'Issue_%': round(issues / total * 100, 2) if total > 0 else 0
    }

# Check emails in each dataset
email_results = []

# Applicant emails
for col in applicant_df.columns:
    if 'email' in col.lower():
        result = validate_emails(applicant_df, col, 'Applicant')
        if result:
            email_results.append(result)

# SEVIS emails
for col in sevis_df.columns:
    if 'email' in col.lower():
        result = validate_emails(sevis_df, col, 'SEVIS')
        if result:
            email_results.append(result)

# Outreach emails
for col in outreach_df.columns:
    if 'email' in col.lower():
        result = validate_emails(outreach_df, col, 'Outreach')
        if result:
            email_results.append(result)

if email_results:
    email_validation_df = pd.DataFrame(email_results)
    print(email_validation_df.to_string(index=False))

    # Highlight issues
    issues_found = email_validation_df[email_validation_df['Potential_Issues'] > 0]
    if len(issues_found) > 0:
        print("\n⚠ EMAIL VALIDATION ISSUES FOUND:")
        for _, row in issues_found.iterrows():
            print(f"  {row['Dataset']} - {row['Email_Column']}: {row['Potential_Issues']} emails without @ symbol")
    else:
        print("\n✓ All email addresses have valid format")
else:
    print("No email columns found for validation")



EMAIL ADDRESS VALIDATION
══════════════════════════════════════════════════════════════════════
  Dataset                      Email_Column  Total_Non_Null  Has_@_Symbol  Has_Valid_Domain  Is_SLU_Email  Potential_Issues  Issue_%
Applicant                          Email_ID            6894          6894              6850            24                 0      0.0
Applicant Official_University_email_address            6831          6831              6831          6831                 0      0.0
    SEVIS                             Email            3848          3848              3848          3840                 0      0.0
 Outreach                      Person_Email            1565          1565              1562             1                 0      0.0
 Outreach          Person_SLU_Email_Address            1565          1565              1565          1565                 0      0.0

✓ All email addresses have valid format


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 8: NUMERIC ANOMALY DETECTION
# ═══════════════════════════════════════════════════════════════════

print("\n\nNUMERIC ANOMALY DETECTION (OUTLIERS)")
print("═" * 70)
print("Using IQR method to identify outliers in financial columns...")

# ───────────────────────────────────────────────────────────────────
# TODO: Detect outliers using IQR (Interquartile Range) method
# Focus on financial columns as they're business-critical
# ───────────────────────────────────────────────────────────────────

def detect_outliers_iqr(df, col, dataset_name):
    """
    Detect outliers using IQR method

    Parameters:
    -----------
    df : pd.DataFrame
        Dataset
    col : str
        Column to check
    dataset_name : str
        Name of dataset

    Returns:
    --------
    dict with outlier statistics or None
    """
    if not pd.api.types.is_numeric_dtype(df[col]):
        return None

    series = df[col].dropna()

    if len(series) == 0:
        return None

    # Calculate IQR
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    # Define outlier boundaries (using 3*IQR for extreme outliers)
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR

    # Find outliers
    outliers = series[(series < lower_bound) | (series > upper_bound)]

    if len(outliers) > 0:
        return {
            'Dataset': dataset_name,
            'Column': col,
            'Total_Values': len(series),
            'Outliers_Count': len(outliers),
            'Outliers_%': round(len(outliers) / len(series) * 100, 2),
            'Min_Value': round(series.min(), 2),
            'Max_Value': round(series.max(), 2),
            'Mean': round(series.mean(), 2),
            'Median': round(series.median(), 2)
        }

    return None

# Focus on financial columns in SEVIS
financial_cols = ['Tuition_Fees', 'Living_Expenses',
                  'Student\'s_Personal_Funds', 'Funds_From_This_School',
                  'Funds_From_Other_Sources']

anomaly_results = []

print("\nAnalyzing SEVIS financial columns...")
for col in financial_cols:
    if col in sevis_df.columns:
        result = detect_outliers_iqr(sevis_df, col, 'SEVIS')
        if result:
            anomaly_results.append(result)

if anomaly_results:
    anomaly_df = pd.DataFrame(anomaly_results)

    print("\n" + "─" * 70)
    print("OUTLIERS DETECTED:")
    print("─" * 70)
    print(anomaly_df.to_string(index=False))

    print("\n💡 NOTE: Outliers are not always errors!")
    print("   - High scholarships (e.g., $76,964 Chess Scholarship) are legitimate")
    print("   - Review outliers in context of business rules")
else:
    print("\n✓ No significant outliers detected in financial columns")



NUMERIC ANOMALY DETECTION (OUTLIERS)
══════════════════════════════════════════════════════════════════════
Using IQR method to identify outliers in financial columns...

Analyzing SEVIS financial columns...

──────────────────────────────────────────────────────────────────────
OUTLIERS DETECTED:
──────────────────────────────────────────────────────────────────────
Dataset                   Column  Total_Values  Outliers_Count  Outliers_%  Min_Value  Max_Value     Mean  Median
  SEVIS             Tuition_Fees          3852            1095       28.43     7055.0    65000.0 25514.06 21000.0
  SEVIS          Living_Expenses          3852             952       24.71        0.0    22100.0 18230.65 18150.0
  SEVIS Student's_Personal_Funds          3852             722       18.74        0.0    95000.0  6970.44     0.0
  SEVIS   Funds_From_This_School          3849             483       12.55        0.0    76964.0  5717.86  1000.0
  SEVIS Funds_From_Other_Sources          3846            

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 9: QUALITY ASSESSMENT SUMMARY & EXPORT
# ═══════════════════════════════════════════════════════════════════

print("\n\n" + "═" * 70)
print("DATA QUALITY ASSESSMENT SUMMARY")
print("═" * 70)

# ───────────────────────────────────────────────────────────────────
# Create comprehensive summary
# ───────────────────────────────────────────────────────────────────

print("\n📊 QUALITY METRICS:")
print("─" * 70)

quality_summary = {
    'Total Datasets Analyzed': 4,
    'Duplicate ID Columns Found': len(duplicate_df[duplicate_df['Duplicate_%'] > 0]),
    'Cross-File Integrity Checks': len(integrity_results),
    'Business Logic Violations': len(business_violations) if business_violations else 0,
    'Email Validation Issues': len(email_results) if email_results else 0,
    'Numeric Outliers Detected': len(anomaly_results) if anomaly_results else 0
}

for metric, value in quality_summary.items():
    print(f"  {metric:.<50} {value}")

# ───────────────────────────────────────────────────────────────────
# Critical Issues Summary
# ───────────────────────────────────────────────────────────────────

print("\n\n🚨 CRITICAL ISSUES REQUIRING ATTENTION:")
print("─" * 70)

critical_issues = []

# Issue 1: Connect duplicates
if len(duplicate_df) > 0:
    max_dup_row = duplicate_df.loc[duplicate_df['Duplicate_%'].idxmax()]
    if max_dup_row['Duplicate_%'] > 50:
        critical_issues.append({
            'Issue': 'High Duplication Rate',
            'Dataset': max_dup_row['Dataset'],
            'Details': f"{max_dup_row['Duplicate_%']:.1f}% duplicate {max_dup_row['ID_Column']}",
            'Impact': 'Inflates metrics, requires deduplication for student-level analysis',
            'Action': 'Investigate if this is event log vs status table'
        })

# Issue 2: Low SEVIS match rate
if len(integrity_results) > 0:
    for check in integrity_results:
        if check['Match_Rate_%'] < 50:
            critical_issues.append({
                'Issue': 'Low Cross-File Match Rate',
                'Dataset': check['Check'],
                'Details': f"Only {check['Match_Rate_%']:.1f}% match",
                'Impact': 'Data synchronization issue or timing mismatch',
                'Action': 'Verify data export dates and sync process'
            })

# Issue 3: Business logic violations
if business_violations:
    for violation in business_violations:
        if violation['Severity'] in ['HIGH', 'CRITICAL']:
            critical_issues.append({
                'Issue': 'Business Logic Violation',
                'Dataset': violation['Rule'],
                'Details': f"{violation['Violations']} violations ({violation['Violation_%']:.1f}%)",
                'Impact': 'Process bottleneck or data entry issues',
                'Action': 'Review workflow and data entry procedures'
            })

# Display critical issues
if critical_issues:
    for i, issue in enumerate(critical_issues, 1):
        print(f"\n{i}. {issue['Issue']}")
        print(f"   Dataset: {issue['Dataset']}")
        print(f"   Details: {issue['Details']}")
        print(f"   Impact:  {issue['Impact']}")
        print(f"   Action:  {issue['Action']}")
else:
    print("\n✓ No critical issues identified")

# ───────────────────────────────────────────────────────────────────
# Export quality reports
# ───────────────────────────────────────────────────────────────────

print("\n\n" + "═" * 70)
print("EXPORTING QUALITY REPORTS")
print("═" * 70)

# Save all quality checks
duplicate_df.to_csv('quality_duplicates.csv', index=False)
print("✓ Saved: quality_duplicates.csv")

integrity_df.to_csv('quality_relationship_integrity.csv', index=False)
print("✓ Saved: quality_relationship_integrity.csv")

if business_violations:
    pd.DataFrame(business_violations).to_csv('quality_business_violations.csv', index=False)
    print("✓ Saved: quality_business_violations.csv")

if email_results:
    email_validation_df.to_csv('quality_email_validation.csv', index=False)
    print("✓ Saved: quality_email_validation.csv")

if anomaly_results:
    anomaly_df.to_csv('quality_numeric_anomalies.csv', index=False)
    print("✓ Saved: quality_numeric_anomalies.csv")

# Save critical issues summary
if critical_issues:
    pd.DataFrame(critical_issues).to_csv('quality_critical_issues.csv', index=False)
    print("✓ Saved: quality_critical_issues.csv")

print("\n" + "═" * 70)
print("NOTEBOOK 2 COMPLETE! ✓")
print("═" * 70)
print("\nQuality Assessment Summary:")
print(f"  • Analyzed {len(duplicate_df)} ID columns for duplicates")
print(f"  • Performed {len(integrity_results)} cross-file integrity checks")
print(f"  • Validated {len(business_violations) if business_violations else 0} business rules")
print(f"  • Found {len(critical_issues)} critical issues requiring attention")
print(f"\nNext: Notebook 3 - Deep Investigations (The 10+ Questions)")

# Download files
from google.colab import files

print("\n" + "─" * 70)
print("DOWNLOAD QUALITY REPORTS")
print("─" * 70)

files.download('quality_duplicates.csv')
files.download('quality_relationship_integrity.csv')
if business_violations:
    files.download('quality_business_violations.csv')
if email_results:
    files.download('quality_email_validation.csv')
if anomaly_results:
    files.download('quality_numeric_anomalies.csv')
if critical_issues:
    files.download('quality_critical_issues.csv')

print("\n✓ All quality reports ready for download!")



══════════════════════════════════════════════════════════════════════
DATA QUALITY ASSESSMENT SUMMARY
══════════════════════════════════════════════════════════════════════

📊 QUALITY METRICS:
──────────────────────────────────────────────────────────────────────
  Total Datasets Analyzed........................... 4
  Duplicate ID Columns Found........................ 4
  Cross-File Integrity Checks....................... 3
  Business Logic Violations......................... 2
  Email Validation Issues........................... 5
  Numeric Outliers Detected......................... 5


🚨 CRITICAL ISSUES REQUIRING ATTENTION:
──────────────────────────────────────────────────────────────────────

1. High Duplication Rate
   Dataset: Connect
   Details: 80.0% duplicate Reference_ID
   Impact:  Inflates metrics, requires deduplication for student-level analysis
   Action:  Investigate if this is event log vs status table

2. Low Cross-File Match Rate
   Dataset: Applicant SEVIS_IDs →

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ All quality reports ready for download!
